In [1]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "zh-CN,zh;q=0.9",
}

url = "http://www.bspider.top/anjuke/"
r = requests.get(url, headers=HEADERS, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding

soup = BeautifulSoup(r.text, "html.parser")
print("状态码:", r.status_code)
print("标题  :", soup.title.get_text(strip=True))
print("是否验证页:", "访问验证" in r.text)
rows = soup.select("div.property")
print("本页房源数:", len(rows))

状态码: 200
标题  : 重庆二手房房产网，重庆二手房出售，重庆买房购房交易信息 - 58安居客
是否验证页: False
本页房源数: 60


In [2]:
row = rows[0]

print("title  :", row.select_one("h3.property-content-title-name").get_text(strip=True))
print("house  :", row.select_one("p.property-content-info-comm-name").get_text(strip=True))

addr_node = row.select_one("p.property-content-info-comm-address")
print("address:", "".join(addr_node.stripped_strings))

attrs = [p.get_text(strip=True) for p in row.select("p.property-content-info-text")]
print("attrs  :", attrs)
print("  户型 =", attrs[0])
print("  面积 =", attrs[1])
print("  朝向 =", attrs[2])
print("  楼层 =", attrs[3])
print("  年代 =", attrs[4])

print("price  :", row.select_one("p.property-price-average").get_text(strip=True))


title  : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house  : 万科金域学府翰林菁华里
address: 南岸学府大道二塘路59号
attrs  : ['3室2厅1卫', '89.9㎡', '南北', '低层(共27层)', '2019年建造']
  户型 = 3室2厅1卫
  面积 = 89.9㎡
  朝向 = 南北
  楼层 = 低层(共27层)
  年代 = 2019年建造
price  : 16465元/㎡


In [3]:
FIELDS = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]

def parse_row(row):
    def txt(sel, default=""):
        node = row.select_one(sel)
        return node.get_text(strip=True) if node else default

    attrs = [p.get_text(strip=True) for p in row.select("p.property-content-info-text")]

    addr_node = row.select_one("p.property-content-info-comm-address")
    address = "".join(addr_node.stripped_strings) if addr_node else ""

    return [
        txt("h3.property-content-title-name"),
        txt("p.property-content-info-comm-name"),
        address,
        attrs[0] if len(attrs) > 0 else "",
        attrs[1] if len(attrs) > 1 else "",
        attrs[3] if len(attrs) > 3 else "",   # 第4项才是楼层
        attrs[4] if len(attrs) > 4 else "",   # 缺失→空字符串
        txt("p.property-price-average"),
    ]

for label, r in [("第1条", rows[0]), ("第2条", rows[1]), ("最后1条", rows[-1])]:
    print(label, dict(zip(FIELDS, parse_row(r))))


第1条 {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
第2条 {'title': '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', 'house': '两代一家', 'address': '渝北渝北黄泥磅紫荆路29号', 'struct': '3室2厅2卫', 'area': '94.5㎡', 'floor': '低层(共28层)', 'maketime': '2008年建造', 'price': '14815元/㎡'}
最后1条 {'title': '精装修 电梯房 4室2厅 91平', 'house': '华润凯旋天地', 'address': '南岸弹子石腾龙大道45号', 'struct': '4室2厅2卫', 'area': '91.3㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15886元/㎡'}


In [4]:
from urllib.parse import urljoin
import time

def get_page(url, tries=4):
    for i in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            r.encoding = r.apparent_encoding
            return r.text
        except Exception as e:
            print(f"  第{i+1}次失败: {type(e).__name__}，重试…")
            time.sleep(2)
    raise RuntimeError("多次重试仍失败: " + url)

current = "http://www.bspider.top/anjuke/"
visited = set()
records = []
page_no = 0
MAX_PAGES = 15

while current and page_no < MAX_PAGES:
    html = get_page(current)
    soup = BeautifulSoup(html, "html.parser")

    page_no += 1
    page_rows = soup.select("div.property")
    for r in page_rows:
        records.append(parse_row(r))
    print(f"第{page_no}页 {current} 本页{len(page_rows)}条 累计{len(records)}条")

    nav = soup.select_one("div.pagination")
    if not nav:
        print("分页区消失 → 结构异常，中止")
        break
    nxt = nav.select_one("a.next.next-active")
    if not nxt or not nxt.get("href"):
        print("下一页按钮已禁用/不存在 → 到达末页，正常停止")
        break

    next_url = urljoin(current, nxt.get("href"))
    if not next_url.startswith("http://www.bspider.top/anjuke/"):
        print("越出教学站，中止:", next_url)
        break
    if next_url in visited:
        print("地址重复，防止死循环:", next_url)
        break

    visited.add(current)
    current = next_url
    time.sleep(1)


第1页 http://www.bspider.top/anjuke/ 本页60条 累计60条
第2页 http://www.bspider.top/anjuke/?page=2 本页60条 累计120条
第3页 http://www.bspider.top/anjuke/?page=3 本页60条 累计180条
第4页 http://www.bspider.top/anjuke/?page=4 本页60条 累计240条
第5页 http://www.bspider.top/anjuke/?page=5 本页60条 累计300条
第6页 http://www.bspider.top/anjuke/?page=6 本页60条 累计360条
第7页 http://www.bspider.top/anjuke/?page=7 本页60条 累计420条
第8页 http://www.bspider.top/anjuke/?page=8 本页60条 累计480条
第9页 http://www.bspider.top/anjuke/?page=9 本页60条 累计540条
第10页 http://www.bspider.top/anjuke/?page=10 本页60条 累计600条
下一页按钮已禁用/不存在 → 到达末页，正常停止


In [5]:
import csv
from pathlib import Path

Path("data").mkdir(exist_ok=True)

seen, unique = set(), []
for rec in records:
    key = tuple(rec)
    if key not in seen:
        seen.add(key)
        unique.append(rec)

print("去重前:", len(records), "去重后:", len(unique), "重复:", len(records) - len(unique))

with open("data/houseinfo.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(FIELDS)
    w.writerows(unique)

with open("data/houseinfo.csv", encoding="utf-8-sig") as f:
    back = list(csv.DictReader(f))
print("回读数据行数(不含表头):", len(back))
for d in (back[0], back[len(back)//2], back[-1]):
    print(d)


去重前: 600 去重后: 498 重复: 102
回读数据行数(不含表头): 498
{'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019年建造', 'price': '16465元/㎡'}
{'title': '精装修洋房 位置安静 全新装修 随时看房', 'house': '协信城立方玺院', 'address': '沙坪坝大学城大学城南二路', 'struct': '3室3厅2卫', 'area': '95.8㎡', 'floor': '中层(共8层)', 'maketime': '2016年建造', 'price': '15658元/㎡'}
{'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015年建造', 'price': '15530元/㎡'}


In [6]:
import csv, pymysql

with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("CSV 表头:", reader.fieldnames)
    csv_rows = [[r[k] for k in FIELDS] for r in reader]
print("CSV 记录数:", len(csv_rows))

conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=False)
cur = None
try:
    cur = conn.cursor()

    cur.execute("SELECT VERSION()")
    print("MySQL 版本:", cur.fetchone()[0])

    cur.execute("SELECT COUNT(*) FROM houseinfo")
    before = cur.fetchone()[0]
    print("入库前条数:", before)

    # 取库里已有的 8 字段组合，用于避免重复插入
    cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = {tuple(r) for r in cur.fetchall()}
    print("库内已有记录:", len(existing))

    to_insert = [r for r in csv_rows if tuple(r) not in existing]
    print("本次待插入:", len(to_insert))

    if to_insert:
        sql = ("INSERT INTO houseinfo "
               "(title, house, address, struct, area, floor, maketime, price) "
               "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")
        affected = cur.executemany(sql, to_insert)
        conn.commit()
        print("executemany 影响行数:", affected)
    else:
        print("无新数据，跳过插入")

    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后条数:", cur.fetchone()[0])
except Exception as e:
    conn.rollback()
    print("出错已回滚:", type(e).__name__, e)
finally:
    if cur is not None:
        cur.close()
    conn.close()
    print("游标与连接已关闭")


CSV 表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 记录数: 498
MySQL 版本: 8.4.11
入库前条数: 0
库内已有记录: 0
本次待插入: 498
executemany 影响行数: 498
入库后条数: 498
游标与连接已关闭


In [7]:
import csv, pymysql

with open("data/houseinfo.csv", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("CSV 表头:", reader.fieldnames)
    csv_rows = [[r[k] for k in FIELDS] for r in reader]
print("CSV 记录数:", len(csv_rows))

conn = pymysql.connect(host="mysql", port=3306, user="student", password="123456",
                       database="smartpark_lab", charset="utf8mb4", autocommit=False)
cur = None
try:
    cur = conn.cursor()

    cur.execute("SELECT VERSION()")
    print("MySQL 版本:", cur.fetchone()[0])

    cur.execute("SELECT COUNT(*) FROM houseinfo")
    before = cur.fetchone()[0]
    print("入库前条数:", before)

    # 取库里已有的 8 字段组合，用于避免重复插入
    cur.execute("SELECT title, house, address, struct, area, floor, maketime, price FROM houseinfo")
    existing = {tuple(r) for r in cur.fetchall()}
    print("库内已有记录:", len(existing))

    to_insert = [r for r in csv_rows if tuple(r) not in existing]
    print("本次待插入:", len(to_insert))

    if to_insert:
        sql = ("INSERT INTO houseinfo "
               "(title, house, address, struct, area, floor, maketime, price) "
               "VALUES (%s, %s, %s, %s, %s, %s, %s, %s)")
        affected = cur.executemany(sql, to_insert)
        conn.commit()
        print("executemany 影响行数:", affected)
    else:
        print("无新数据，跳过插入")

    cur.execute("SELECT COUNT(*) FROM houseinfo")
    print("入库后条数:", cur.fetchone()[0])
except Exception as e:
    conn.rollback()
    print("出错已回滚:", type(e).__name__, e)
finally:
    if cur is not None:
        cur.close()
    conn.close()
    print("游标与连接已关闭")


CSV 表头: ['title', 'house', 'address', 'struct', 'area', 'floor', 'maketime', 'price']
CSV 记录数: 498
MySQL 版本: 8.4.11
入库前条数: 498
库内已有记录: 498
本次待插入: 0
无新数据，跳过插入
入库后条数: 498
游标与连接已关闭
